# Advanced Certification Programme in Agentic and Generative AI
## A Programme by IISc and TalentSprint
## Assignment Notebook: Evaluation Metrics

## Learning Objectives

The objective of this notebook is to provide hands-on experience in evaluating LLM outputs using both traditional NLP metrics and modern LLM-based evaluation techniques.

In this notebook, you will:

* Understand the importance of evaluation in Generative AI applications.
* Learn how to measure text generation quality using **BLEU**, **ROUGE**, and **METEOR** metrics.
* Explore semantic similarity evaluation using **BERTScore**.
* Compare lexical-based metrics with embedding-based evaluation approaches.
* Implement and analyze **LLM-as-a-Judge** for qualitative assessment of model outputs.
* Interpret evaluation scores and understand the strengths and limitations of different metrics.


## Information

**Evaluation** plays a critical role in assessing the performance and reliability of language models.

In this assignment, learners will explore both automatic and human-centric evaluation methods to understand their strengths and limitations. By implementing classical metrics **(BLEU, ROUGE, METEOR, BERTScore)** and integrating **LLM-as-a-judfe**, participants will gain practical insights into designing robust evaluation pipelines. This hands-on task also highlights why human judgment remains essential despite the growing use of automated evaluation frameworks.

Although metrics like BLEU and METEOR are automatically computed, they rely on human-authored reference outputs as benchmarks. This makes them indirect reflections of human-centric evaluation. The human element comes from curating reference answers, while the automated metrics quantify similarity between model responses and those human references.

### **Setup Steps:**

In [ ]:
#@title Please enter your registration id to start: { run: "auto", display-mode: "form" }
Id = "" #@param {type:"string"}

In [ ]:
#@title Please enter your password (your registered phone number) to continue: { run: "auto", display-mode: "form" }
password = "" #@param {type:"string"}

In [ ]:
#@title Run this cell to complete the setup for this Notebook
from IPython import get_ipython
from IPython.display import HTML, display

ipython = get_ipython()

notebook= "M8_AST_01_Evaluation_Metrics"   #name of the notebook

batchId = "IISC-AC-GENAI-02"


def print_message(message: str, color: str = "red"):
    display(HTML(f"<span style='color:{color};'>{message}</span>"))

def setup():
#  ipython.magic("sx pip3 install torch")

    from IPython.display import HTML, display
    display(HTML('<script src="https://dashboard.talentsprint.com/submissions/record_ip.html?traineeId={0}&recordId={1}"></script>'.format(getId(),submission_id)))
    print("Setup completed successfully")
    return

def submit_notebook():
    ipython.magic("notebook -e "+ notebook + ".ipynb")

    import requests, json, base64, datetime

    url = "https://dashboard.talentsprint.com/xp/app/save_notebook_attempts"
    if not submission_id:
      data = {"id" : getId(), "notebook" : notebook, "mobile" : getPassword(), "batch" : batchId}
      r = requests.post(url, data = data)
      r = json.loads(r.text)

      if r["status"] == "Success":
          return r["record_id"]
      elif "err" in r:
        print_message(r["err"])
        return None
      else:
        print_message("Something is wrong, the notebook will not be submitted for grading")
        return None

    elif getAnswer() and getComplexity() and getAdditional() and getConcepts() and getComments() and getMentorSupport():
      f = open(notebook + ".ipynb", "rb")
      file_hash = base64.b64encode(f.read())

      data = {"complexity" : Complexity, "additional" :Additional,
              "concepts" : Concepts, "record_id" : submission_id,
              "answer" : Answer, "id" : Id, "file_hash" : file_hash,
              "notebook" : notebook,
              "feedback_experiments_input" : Comments,
              "feedback_mentor_support": Mentor_support,
              "batch" : batchId
            }
      r = requests.post(url, data = data)
      r = json.loads(r.text)
      if "err" in r:
        print(r["err"])
        return None
      else:
        print("Your submission is successful.")
        print("Ref Id:", submission_id)
        print("Date of submission: ", r["date"])
        print("Time of submission: ", r["time"])
        print("View your submissions: https://learn-iisc.talentsprint.com/notebook_submissions")
        #print("For any queries/discrepancies, please connect with mentors through the chat icon in LMS dashboard.")
        return submission_id
    else: submission_id


def getAdditional():
  try:
    if not Additional:
      raise NameError
    else:
      return Additional
  except NameError:
    print ("Please answer Additional Question")
    return None

def getComplexity():
  try:
    if not Complexity:
      raise NameError
    else:
      return Complexity
  except NameError:
    print ("Please answer Complexity Question")
    return None

def getConcepts():
  try:
    if not Concepts:
      raise NameError
    else:
      return Concepts
  except NameError:
    print ("Please answer Concepts Question")
    return None


# def getWalkthrough():
#   try:
#     if not Walkthrough:
#       raise NameError
#     else:
#       return Walkthrough
#   except NameError:
#     print ("Please answer Walkthrough Question")
#     return None

def getComments():
  try:
    if not Comments:
      raise NameError
    else:
      return Comments
  except NameError:
    print ("Please answer Comments Question")
    return None


def getMentorSupport():
  try:
    if not Mentor_support:
      raise NameError
    else:
      return Mentor_support
  except NameError:
    print ("Please answer Mentor support Question")
    return None

def getAnswer():
  try:
    if not Answer:
      raise NameError
    else:
      return Answer
  except NameError:
    print ("Please answer Question")
    return None


def getId():
  try:
    return Id if Id else None
  except NameError:
    return None

def getPassword():
  try:
    return password if password else None
  except NameError:
    return None

submission_id = None
### Setup
if getPassword() and getId():
  submission_id = submit_notebook()
  if submission_id:
    setup()
else:
  print ("Please complete Id and Password cells before running setup")

### Install Required Packages

In [ ]:
!pip -q install rouge-score   # Install ROUGE metric library (used for evaluating text summarization and generation quality)
!pip -q install bert_score    # Install BERTScore (semantic similarity evaluation using contextual embeddings)

!pip -q install nltk          # Install NLTK (Natural Language Toolkit for tokenization, preprocessing, etc.)
!pip -q install sacrebleu     # Install SacreBLEU (standardized BLEU score implementation for text generation evaluation)

!pip -q install transformers  # Install Hugging Face Transformers (to use pretrained models like BERT, GPT, etc.)
!pip -q install datasets      # Install Hugging Face Datasets (easy access to benchmark datasets and evaluation corpora)

# Install the official OpenAI Python SDK
# - Provides API access to OpenAI models (e.g., GPT models) used for generating completions, embeddings, chat responses, etc.
!pip -q install openai

### Import Required Packages

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from openai import OpenAI
from datasets import load_dataset, Dataset
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate import meteor_score
from nltk.tokenize import word_tokenize

### **Read the OpenAI API Key**

* Use an existing OpenAI API key, or Create a new one by visiting: https://platform.openai.com/api-keys

* Save the key in Google Colab's Secrets

* Read the key and save as an environment variable `OPENAI_API_KEY`

In [ ]:
# Save the key in Colab's Secrets then load from there

from google.colab import userdata

# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')           # In using OpenAI model

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')               # In using Groq model

### 1: **Create OpenAI client**

In [ ]:
# Initialize OpenAI client
# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))            # In using OpenAI model

# Initialize OpenAI client with Groq base URL
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),                          # In using Groq model
    base_url="https://api.groq.com/openai/v1"
)

### 2: Load Small Evaluation Dataset

In [ ]:
eval_data = [
    {"id":"q1",
     "prompt":"What is the baggage allowance for international flights? Respond in 2 sentences.",
     "reference":"Each passenger is allowed 1 checked bag up to 23kg and 1 carry-on."
     },
    {"id":"q2",
     "prompt":"How do I change my booking date? Respond in 2 sentences.",
     "reference":"You can change the booking on the website under 'Manage Booking' or contact support; change fees may apply."
     },
    {"id":"q3",
     "prompt":"Can I get a refund for my cancelled ticket? Respond in 2 sentences.",
     "reference":"Refunds depend on the fare type; refundable tickets can be processed via your account, otherwise contact support."
     },
    {"id":"q4",
     "prompt":"How early should I arrive at the airport for a domestic flight? Respond in 2 sentences.",
     "reference":"It is recommended to arrive at least 2 hours before departure for domestic flights."
     },
    {"id":"q5",
     "prompt":"Do you provide special meals on flights? Respond in 2 sentences.",
     "reference":"Yes, special meals can be requested during booking or 24 hours before departure."
     },
    {"id":"q6",
     "prompt":"How can I upgrade to business class? Respond in 2 sentences.",
     "reference":"You can upgrade online via 'Manage Booking', at check-in, or contact customer support."
     },
    {"id":"q7",
     "prompt":"Is my baggage automatically transferred during connecting flights? Respond in 2 sentences.",
     "reference":"Yes, for flights on the same ticket, baggage is usually transferred automatically. Confirm at check-in."
     },
    {"id":"q8",
     "prompt":"What is your policy for traveling with pets? Respond in 2 sentences.",
     "reference":"Pets can travel in the cabin or as checked baggage; you must inform the airline during booking and follow size/weight limits."
     },
    {"id":"q9",
     "prompt":"How do I apply for a visa assistance letter? Respond in 2 sentences.",
     "reference":"Visa assistance letters can be requested from customer support; provide flight details and personal info."
     },
    {"id":"q10",
     "prompt":"Can I reserve seats in advance? Respond in 2 sentences.",
     "reference":"Yes, seat selection is available during booking or via 'Manage Booking'; fees may apply for certain seats."
     }
]

In [ ]:
type(eval_data[0])

In [ ]:
# Convert a list of dictionaries (eval_data) into a Hugging Face Dataset object
# This makes the data compatible with Hugging Face utilities for evaluation,
# batching, mapping functions, and metric computation.
ds = Dataset.from_list(eval_data)
ds

In [ ]:
ds[0]

### 3: Generate Model Outputs

In [ ]:
MODEL = "llama-3.1-8b-instant"           # If using model from Groq
# MODEL = "gpt-3.5-turbo"                 # If using OpenAI model


def generate_answer(prompt, model=MODEL, max_tokens=150):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0  # deterministic output for evaluation
    )
    return response.choices[0].message.content.strip()


In [ ]:
# Run generation for all evaluation dataset items

preds = []

for item in ds:
    answer = generate_answer(item['prompt'])        # Get LLM response

    # Sleep for 25 seconds in each iteration to adhere to Rate Limits of the model
    import time
    time.sleep(25)

    preds.append({                                  # Append to the `preds` list
        "id": item['id'],
        "prompt": item['prompt'],
        "reference": item['reference'],
        "prediction": answer
    })


In [ ]:
preds

The above variable 'preds' now contains model outputs ready for evaluation.

### 4: Save Predictions in a File (for reproducibility)

The following code cell is saving the predictions.
- LLM outputs are often non-deterministic (due to temperature, sampling, or backend randomness).
- Saving predictions ensures that you can re-run evaluation metrics later (BLEU, ROUGE, METEOR, BERTScore) without re-generating outputs.
- This is crucial for consistent comparison across models, experiments, or parameter settings.

In [ ]:
# Open (or create) a file named 'predictions.jsonl' in write mode
# 'w' → overwrite file if it already exists
# encoding='utf-8' → ensures proper handling of non-English characters
with open('predictions.jsonl', 'w', encoding='utf-8') as f:

    # Iterate over each prediction dictionary in the list 'preds'
    for r in preds:

        # Convert the dictionary to a JSON string
        # ensure_ascii=False → keeps Unicode characters readable (e.g., Hindi, accents)
        # '\n' → write each JSON object on a new line (JSONL format)
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

# Confirmation message after saving the file
print("Saved predictions.jsonl")

### 5: Compute BLEU (sacrebleu)

**BLEU**

It stands for **Bilingual Evaluation Understudy**, is a traditional NLP evaluation metric used to measure how closely a machine-generated text matches one or more human-written reference texts.

It is commonly used in tasks such as **machine translation**, **text generation**, and **summarization**, where the output generated by an AI model needs to be compared with an expected reference answer.

---

**How it works?**

BLEU works by checking the overlap of words or word sequences, called **n-grams**, between the generated output and the reference text. For example, it can compare single words, two-word phrases, three-word phrases, and so on. A higher overlap generally results in a higher BLEU score.

The BLEU score usually ranges from **0 to 1**, or sometimes from **0 to 100**. A higher score indicates that the generated response is more similar to the reference response.

---

**Limitations**

It mainly focuses on exact word matching, so it may give a low score even when the generated answer is meaningful but uses different wording.

For example, *“The customer cancelled the booking”* and *“The booking was called off by the customer”* may have similar meaning, but BLEU may not fully capture this semantic similarity.

In short, **BLEU is useful for measuring lexical similarity, but it should not be used alone to judge the quality of LLM outputs**, especially when meaning, reasoning, fluency, and factual correctness are important.

---

This following code cell extracts the ground-truth reference texts and the model-generated predictions (hypotheses) from the saved results list (`preds`).
- Most evaluation metrics like BLEU, ROUGE, METEOR and BERTScore require these two lists as inputs to compare what the model produced versus the expected output.
- By preparing **refs** and **hyps** in clean, aligned lists, it ensures each prediction is matched correctly to its reference before score computation.

In [ ]:
preds

In [ ]:
# Load predictions from the saved file if they are not already present in memory
# 'locals()' checks whether the variable 'preds' already exists in the current scope
if 'preds' not in locals():

    preds = []    # Initialize an empty list to store prediction records

    # Open the JSONL file in read mode with UTF-8 encoding
    with open('predictions.jsonl', 'r', encoding='utf-8') as f:

        # Read the file line by line
        for line in f:

            # Convert each JSON string line back into a Python dictionary
            # and append it to the preds list
            preds.append(json.loads(line))


# Initialize empty lists to store references and hypotheses separately
refs = []
hyps = []

# Extract reference and prediction text from each record
for item in preds:
    refs.append(item['reference'])         # Append the ground-truth reference text
    hyps.append(item['prediction'])        # Append the generated model prediction

In [ ]:
# Ground Truths
refs

In [ ]:
# LLM responses
hyps

In [ ]:
# Compute corpus-level BLEU score
# corpus_bleu evaluates overall translation/text generation quality
# across the entire dataset (not per sentence).
# hyps → list of generated texts
# [refs] → sacrebleu expects references as a list of reference lists
bleu = sacrebleu.corpus_bleu(hyps, [refs])

# bleu.score returns the final BLEU value (scaled between 0–100)
print("Corpus BLEU Score:", bleu.score)

In [ ]:
# Compute sentence-level BLEU score for each example
# sentence_bleu calculates BLEU for one (hypothesis, reference) pair at a time
# This is mainly for illustration or detailed analysis

bleu_scores = []

# Iterate over each reference (r) and hypothesis (h) pair
for r, h in zip(refs, hyps):
    bleu_scores.append(
        sacrebleu.sentence_bleu(h, [r]).score    # Compare single prediction with its reference
    )

# Print BLEU score for each individual example
print("Smoothed Sentence BLEU Scores (per example, for illustration):\n", bleu_scores)

### 6: Compute ROUGE (rouge_scorer)

**ROUGE**

It stands for **Recall-Oriented Understudy for Gisting Evaluation**, is a traditional NLP evaluation metric used to compare a machine-generated text with one or more human-written reference texts.

ROUGE is commonly used for evaluating **summarization**, **text generation**, and other tasks where the goal is to check how much important information from the reference answer is captured in the generated output.

---

**How it works?**

Unlike BLEU, which focuses more on precision, **ROUGE mainly focuses on recall**. This means it checks how much of the reference text is covered by the generated response.

Some commonly used ROUGE variants are:

* **ROUGE-1**: Measures overlap of individual words between the generated output and reference text.
* **ROUGE-2**: Measures overlap of two-word sequences.
* **ROUGE-L**: Measures the longest common sequence of words between the generated and reference texts.

A higher ROUGE score indicates that the generated output contains more content similar to the reference answer.

---

**Limitations**

It relies heavily on word overlap and may not fully capture semantic meaning. For example, two answers may express the same idea using different words, but ROUGE may give a lower score because the exact words do not match.

In short, **ROUGE is useful for checking content coverage, especially in summarization tasks, but it should be combined with semantic and human/LLM-based evaluation methods for a more complete assessment of LLM outputs**.

---

In [ ]:
# Initialize empty lists to store ROUGE F-measure scores
# F-measure (F1) balances precision and recall
# rouge1 → unigram overlap (single words)
# rouge2 → bigram overlap (two consecutive words)
# rougeL → longest common subsequence (sequence similarity)
rouge1_f, rouge2_f, rougel_f = [], [], []

# Initialize empty lists to store ROUGE Recall scores
# Recall measures how much of the reference text is captured by the generated text
rouge1_r, rouge2_r, rougel_r = [], [], []

In [ ]:
# Create a ROUGE scorer object
# ['rouge1', 'rouge2', 'rougeL'] → specify which ROUGE variants to compute
# use_stemmer=True → applies stemming (e.g., the word "running" is converted to → "run") to improve matching

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)


In [ ]:
# Iterate over each reference (r) and hypothesis (h) pair
for r, h in zip(refs, hyps):

    # Compute ROUGE scores for the pair
    scores = scorer.score(r, h)

    # Append F-measure values
    rouge1_f.append(scores['rouge1'].fmeasure)
    rouge2_f.append(scores['rouge2'].fmeasure)
    rougel_f.append(scores['rougeL'].fmeasure)

    # Append Recall values
    rouge1_r.append(scores['rouge1'].recall)
    rouge2_r.append(scores['rouge2'].recall)
    rougel_r.append(scores['rougeL'].recall)


# Compute and print average (mean) ROUGE scores across all examples
print("ROUGE-1 F:", np.mean(rouge1_f))
print("ROUGE-1 R:", np.mean(rouge1_r))
print("ROUGE-2 F:", np.mean(rouge2_f))
print("ROUGE-2 R:", np.mean(rouge2_r))
print("ROUGE-L F:", np.mean(rougel_f))
print("ROUGE-L R:", np.mean(rougel_r))

### 7: Compute METEOR (NLTK)

**METEOR**

It stands for **Metric for Evaluation of Translation with Explicit ORdering**, is an NLP evaluation metric used to compare machine-generated text with human-written reference text.

It is commonly used for tasks such as **machine translation**, **text generation**, and **summarization**.

METEOR improves on metrics like BLEU by considering not only exact word matches, but also:

* **Synonyms**
* **Word stems**
* **Similar word forms**
* **Word order**

For example, words like **“run,” “running,” and “ran”** may be treated as related, and words with similar meanings may receive partial credit. This makes METEOR more flexible than purely lexical overlap-based metrics.

---

**How it works?**

METEOR calculates a score based on both **precision** and **recall**. Precision checks how much of the generated output is relevant, while recall checks how much of the reference answer is covered by the generated output.

A higher METEOR score indicates that the generated text is more similar to the reference text in terms of meaning and word usage.

---

**Limitations**

It still depends on reference answers and may not fully capture deeper reasoning, factual correctness, or context-specific meaning.

In short, **METEOR provides a more flexible evaluation than BLEU by considering synonyms and word variations, but it should still be combined with semantic and human/LLM-based evaluation methods for reliable assessment of LLM outputs**.


In [ ]:
word_tokenize("This is a test sentence")

In [ ]:
# Compute METEOR score for each (reference, hypothesis) pair
# METEOR evaluates text generation quality using word overlap,
# stemming, synonym matching, and word order alignment.

meteor_scores = []

for r, h in zip(refs, hyps):         # Iterate pairwise over references and predictions
    meteor_scores.append(
        # Tokenize both reference (r) and hypothesis (h) into words
        # `word_tokenize` ensures proper splitting instead of naive .split()
        meteor_score.single_meteor_score(
            word_tokenize(r),   # Reference tokens
            word_tokenize(h)    # Generated text tokens
        )
    )

# Compute and print the average METEOR score across all examples
print("METEOR Score:", sum(meteor_scores) / len(meteor_scores))

#### Demonstrating METEOR's Synonym Handling

METEOR considers synonyms, which can lead to higher scores compared to metrics like BLEU and ROUGE when the generated text uses different but related words than the reference. Here's an example:

In [ ]:
# Define a reference sentence (ground truth)
reference = "The quick brown fox jumps over the lazy dog."

# Define a generated sentence (model output) for comparison
hypothesis = "A swift brown fox leaps over the lethargic dog."


# -------------------- METEOR --------------------

# Tokenize both sentences into word-level tokens
# Proper tokenization is required for METEOR computation
reference_tokens = word_tokenize(reference)
hypothesis_tokens = word_tokenize(hypothesis)

# Compute METEOR score
# METEOR considers:
# - Exact word matches
# - Stemming (jump vs jumps)
# - Synonyms (quick vs swift, lazy vs lethargic)
# - Word order penalty
meteor = meteor_score.single_meteor_score(reference_tokens, hypothesis_tokens)

# Print METEOR score rounded to 4 decimal places
print(f"METEOR Score: {meteor:.4f}")


# -------------------- BLEU --------------------

# Compute sentence-level BLEU score
# BLEU measures n-gram overlap (precision-based)
# Does NOT account for synonyms unless words match exactly
bleu = sacrebleu.sentence_bleu(hypothesis, [reference]).score

# Print BLEU score (ranges 0–100 in SacreBLEU)
print(f"BLEU Score: {bleu / 100:.4f}")


# -------------------- ROUGE-L --------------------

# Initialize ROUGE scorer for ROUGE-L (Longest Common Subsequence)
# use_stemmer=True allows matching similar word forms
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# Compute ROUGE-L F1 score
# ROUGE-L measures longest matching word sequence between sentences
rougeL = scorer.score(reference, hypothesis)['rougeL'].fmeasure

# Print ROUGE-L F1 score (range 0–1)
print(f"ROUGE-L F1 Score: {rougeL:.4f}")

### 8: Compute BERTScore (bert_score)

**BERTScore**

It is a modern NLP evaluation metric used to measure the **semantic similarity** between machine-generated text and human-written reference text.

---

**How it works?**

Unlike BLEU, ROUGE, and METEOR, which mainly depend on word overlap, BERTScore uses **contextual embeddings** from transformer-based models like BERT. This allows it to understand the meaning of words based on their context.

For example, if the reference sentence is:

**“The customer cancelled the booking.”**

and the generated sentence is:

**“The customer called off the reservation.”**

Traditional metrics may give a lower score because the exact words are different. However, BERTScore can recognize that **“cancelled”** and **“called off”**, and **“booking”** and **“reservation”**, are semantically similar.

BERTScore calculates three main values:

* **Precision**: How much of the generated text is semantically relevant to the reference.
* **Recall**: How much of the reference meaning is captured by the generated text.
* **F1 Score**: A balanced score combining precision and recall.

A higher BERTScore indicates that the generated output is closer in meaning to the reference output.

---

**Limitations**

It depends on the quality of the underlying language model and may not always detect factual errors, reasoning mistakes, or task-specific correctness.

In short, **BERTScore is useful for evaluating meaning-based similarity and is more flexible than word-overlap metrics, but it should still be combined with human review or LLM-as-a-Judge for a complete evaluation of LLM outputs**.

---

In [ ]:
# Compute BERTScore between generated texts (hyps) and reference texts (refs)
# lang='en' → language of evaluation
# model_type="roberta-large" → pretrained model used to compute semantic similarity
# idf=True → applies inverse document frequency weighting for better importance scaling
# rescale_with_baseline=True → rescales scores for improved interpretability

P, R, F1 = bert_score(
    hyps,
    refs,
    lang='en',
    model_type="roberta-large",
    idf=True,
    rescale_with_baseline=True
)

# It returns three tensors (or arrays):
# P → Precision scores
# R → Recall scores
# F1 → F1 scores

In [ ]:
# Precision scores
P

In [ ]:
# Recall scores
R

In [ ]:
# F1 scores
F1

In [ ]:
# Convert tensor outputs to standard Python float values
# (bert_score returns PyTorch tensors by default)
F1 = [float(x) for x in F1]

# Compute the average (mean) F1 score across all samples
# np.mean calculates the overall semantic similarity score
F1_mean = float(np.mean(F1))

# Print the final aggregated BERTScore F1 value
print("BERTScore F1 (mean):", F1_mean)

### 9: Aggregate & Show table

In [ ]:
# Create a Pandas DataFrame from the list of prediction dictionaries
# Each row corresponds to one evaluated example
df = pd.DataFrame(preds)

df.head()

In [ ]:
# Add sentence-level BLEU scores to the DataFrame
df['bleu'] = bleu_scores

# Normalize BLEU score from 0–100 scale to 0–1 scale
# This makes it comparable with other metrics like ROUGE, METEOR, BERTScore
df['bleu_normalized'] = df['bleu'] / 100

# Add ROUGE recall scores (coverage of reference text)
df['rouge1'] = rouge1_r   # Unigram recall
df['rouge2'] = rouge2_r   # Bigram recall
df['rougeL'] = rougel_r   # Longest Common Subsequence recall

# Add METEOR score (accounts for stemming, synonyms, word order)
df['meteor'] = meteor_scores

# Add BERTScore F1 (semantic similarity using contextual embeddings)
df['bertscore_f1'] = F1

# Display the first 10 rows with selected columns
# Shows prompt, prediction, reference, and all evaluation metrics together
df[['id','prompt','reference','prediction',
    'bleu','bleu_normalized',
    'rouge1','rouge2','rougeL',
    'meteor','bertscore_f1']].head(10)

**Comparative Visualization**

In [ ]:
# Define the list of evaluation metrics to visualize
# All metrics are scaled between 0 and 1 (BLEU is normalized earlier)
metrics = ['bleu_normalized', 'rouge1', 'rouge2', 'rougeL', 'meteor', 'bertscore_f1']

# Extract example IDs from the DataFrame (used as x-axis labels)
ids = df['id']

# Create numeric positions for each ID on the x-axis
x = np.arange(len(ids))  # [0, 1, 2, ..., n-1]

# Set the width of each bar
# Since multiple metrics are plotted per ID, small width avoids overlap
width = 0.12

# Create a figure and axis object with specified size
fig, ax = plt.subplots(figsize=(15, 6))

# Loop through each metric and plot its bars
for i, metric in enumerate(metrics):

    # Shift each metric slightly on the x-axis so bars appear side-by-side
    ax.bar(x + i * width, df[metric], width, label=metric)


ax.set_xlabel('ID')                          # Set x-axis label
ax.set_ylabel('Score')                       # Set y-axis label
ax.set_title('Comparative Metrics per ID')   # Set chart title
ax.set_xticks(x + width * 2)                 # Center x-axis ticks under grouped bars
ax.set_xticklabels(ids, rotation=45)         # Set ID labels and rotate for readability
ax.legend()                                  # Display legend to identify each metric
plt.tight_layout()                           # Adjust layout to prevent label cutoff
plt.show()                                   # Display the plot

In the above result,

- The ROUGE-2 score evaluates the overlap of bigrams (pairs of consecutive words) between a reference text and a generated response. In cases where the generated answer and the reference convey similar meaning **but do not share any exact two-word sequences**, the ROUGE-2 score becomes zero, even if some individual words overlap. This happens because ROUGE-2 strictly counts matching bigrams, not semantic similarity.

- ROUGE-1, however, remains non-zero in such situations because it measures overlap at the single-word level. Even if multi-word phrases do not match exactly, shared individual words can still contribute to a positive ROUGE-1 score.

### **10: LLM-as-a-Judge**

**LLM-as-a-Judge**

It is a modern evaluation approach where a large language model is used to assess the quality of another model’s output.

Instead of relying only on word overlap or reference-based metrics like BLEU, ROUGE, METEOR, or BERTScore, LLM-as-a-Judge evaluates responses based on broader quality criteria such as:

* **Correctness**
* **Relevance**
* **Completeness**
* **Clarity**
* **Coherence**
* **Helpfulness**
* **Factual accuracy**

---

**How it works?**

In this approach, the evaluator LLM is given the user question, the generated response, and sometimes a reference answer or scoring rubric. It then provides a score, rating, or explanation for how good the generated response is.

For example, if the task is to answer a customer query, the judge model can check whether the response directly answers the question, uses a professional tone, includes all required details, and avoids incorrect information.

LLM-as-a-Judge is especially useful for evaluating **open-ended LLM outputs**, where there may be more than one correct answer. It can capture aspects like reasoning quality, tone, and usefulness, which traditional metrics may miss.

---

**Limitations**

The judge model may be biased, inconsistent, or overly influenced by prompt wording. It may also make mistakes while evaluating factual correctness. Therefore, it is important to use clear rubrics, multiple test cases, and human review for critical evaluations.

In short, **LLM-as-a-Judge is useful for qualitative and flexible evaluation of LLM outputs, but it should be combined with traditional metrics and human judgment for reliable assessment**.

---

In [ ]:
# Define a prompt template for the LLM-as-a-Judge setup
# The model is instructed to compare reference and generated text
# and output ONLY a numeric score between 0.000000 and 1.000000
prompt_template = '''
Compare the Original reference with the generated response based on the following rubric:

1. Relevance (0-0.4): How well does the generated response directly address the user's prompt? Score higher for more direct and relevant answers.
2. Completeness (0-0.3): Does the generated response include all the key information present in the original reference? Score higher for including more details from the reference.
3. Fluency (0-0.3): Is the generated response well-written, grammatically correct, and easy to understand? Score higher for better readability and structure.

Provide a single score between 0.000000 and 1.000000, which is the sum of the scores from each criterion. Just give the score value. No additional text.

Original reference: {reference}
Generated response: {answer}
'''


# List to store LLM-generated evaluation scores
score_from_judge_LLM = []


JUDGE_LLM = 'llama-3.1-8b-instant'                  # If using Groq model
# JUDGE_LLM = 'gpt-4o-mini'                   # If using OpenAI model


# Iterate through each prediction example
for item in preds:

    # Format the prompt by inserting reference and generated response
    formatted_prompt = prompt_template.format(
        reference=item['reference'],
        answer=item['prediction']
    )

    # Invoke the judge LLM to evaluate similarity
    resp = client.chat.completions.create(
        model=JUDGE_LLM,                                             # <-----Judge LLM
        messages=[{"role": "user", "content": formatted_prompt}],
        max_tokens=150,
        temperature=0.0
    )

    # Append the numeric score returned by the model
    score_from_judge_LLM.append(resp.choices[0].message.content.strip())

    # Sleep for 25 seconds in each iteration to adhere to Rate Limits of the model
    import time
    time.sleep(25)



In [ ]:
print("Score from judge LLM:\n", score_from_judge_LLM)

In [ ]:
# Convert each LLM judge score (currently stored as a string)
# into a float value for numerical analysis

# score.strip() → removes any leading/trailing spaces or newline characters
# float(...) → converts the cleaned string into a numeric float

scores_from_judge_LLM = []

for score in score_from_judge_LLM:
    scores_from_judge_LLM.append(float(score.strip()))

# Display the cleaned list of numeric scores
scores_from_judge_LLM

In [ ]:
# Create a new DataFrame from the original predictions list
# Each row corresponds to one evaluated example
df_LLM_as_judge = pd.DataFrame(preds)

df_LLM_as_judge.head()

In [ ]:
# Add sentence-level BLEU scores (0–100 scale)
df_LLM_as_judge['bleu'] = bleu_scores

# Normalize BLEU score to 0–1 range for consistency with other metrics
# NOTE: This uses df['bleu'], assuming it was computed earlier.
# Ideally, it should reference df_LLM_as_judge['bleu'] for clarity.
df_LLM_as_judge['bleu_normalized'] = df['bleu'] / 100

# Add ROUGE recall scores
# rouge1 → unigram recall
# rouge2 → bigram recall
# rougeL → longest common subsequence recall
df_LLM_as_judge['rouge1'] = rouge1_r
df_LLM_as_judge['rouge2'] = rouge2_r
df_LLM_as_judge['rougeL'] = rougel_r

# Add METEOR score (accounts for stemming, synonyms, and word order)
df_LLM_as_judge['meteor'] = meteor_scores

# Add BERTScore F1 (semantic similarity using contextual embeddings)
df_LLM_as_judge['bertscore_f1'] = F1

# Add scores generated by the LLM-as-a-Judge evaluation
# These are semantic similarity scores produced by a separate LLM
df_LLM_as_judge['score_from_judge_LLM'] = scores_from_judge_LLM

# Display first 10 rows with selected columns for inspection
df_LLM_as_judge[['id','prompt','reference','prediction',
                 'bleu','bleu_normalized',
                 'rouge1','rouge2','rougeL',
                 'meteor','bertscore_f1',
                 'score_from_judge_LLM']].head(10)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# List of evaluation metrics to visualize
# Includes rule-based, embedding-based, and LLM-as-a-Judge scores
metrics = ['bleu_normalized', 'rouge1', 'rouge2', 'rougeL', 'meteor', 'bertscore_f1', 'score_from_judge_LLM']

# Create a safe copy of the evaluation DataFrame
# This prevents accidental modification of the original dataset
df_plot = df_LLM_as_judge.copy()

# Ensure all metric columns:
# - Are numeric
# - Replace invalid values with 0.0
# - Are clipped within the valid range [0, 1]
for m in metrics:
    df_plot[m] = (
        pd.to_numeric(df_plot[m], errors='coerce')  # convert to numeric, invalid → NaN
        .fillna(0.0)                                # replace NaN with 0.0
        .clip(0.0, 1.0)                             # restrict values between 0 and 1
    )

# Convert IDs to string format for proper x-axis labeling
ids = df_plot['id'].astype(str)

# Create numeric positions for each ID
x = np.arange(len(ids))  # positions like [0, 1, 2, ..., n-1]

# Width of each bar (small since many metrics are plotted side-by-side)
width = 0.12

# Create figure and axis object
fig, ax = plt.subplots(figsize=(15, 6))

# Plot grouped bars for each metric
for i, metric in enumerate(metrics):
    # Shift bars horizontally to prevent overlap
    ax.bar(x + i * width, df_plot[metric], width, label=metric)

# Set axis labels
ax.set_xlabel('ID')
ax.set_ylabel('Score')

# Set chart title
ax.set_title('Comparative Metrics per ID')

# Center x-axis ticks under grouped bars
ax.set_xticks(x + width * (len(metrics) - 1) / 2)
ax.set_xticklabels(ids, rotation=45)

# Fix y-axis scale from 0 to 1 (since all metrics are normalized)
ax.set_ylim(0.0, 1.0)

# Disable autoscaling to maintain consistent scale
ax.set_autoscale_on(False)

# Set evenly spaced y-axis ticks (0.0 to 1.0 with step 0.1)
ax.set_yticks(np.linspace(0.0, 1.0, 11))

# Place legend outside the plot area for better readability
ax.legend(ncol=1, bbox_to_anchor=(1.02, 1), loc='upper left')

# Adjust layout to prevent clipping
plt.tight_layout()

# Display the final visualization
plt.show()

From the above result, we can observe how metrics change when an LLM is used as the judge.

#### **Decision Recipe: Choosing the Right Evaluation**

Selecting the appropriate evaluation method depends on your specific goals and the nature of your LLM application. Here's a brief guide based on the metrics and approaches explored in this notebook:

*   **Traditional Metrics (BLEU, ROUGE, METEOR, BERTScore):**
    *   **Use when:** You have reliable human-authored reference responses and need to measure lexical or semantic similarity. These are good for initial assessments and tracking progress on tasks like summarization, translation, or question answering where a single correct answer exists or can be well-represented by a reference.
    *   **Considerations:** These metrics are sensitive to exact word choices and sentence structure (especially BLEU and ROUGE). METEOR and BERTScore offer more flexibility by considering synonyms and semantic meaning. They do not directly measure factual correctness or overall helpfulness from a human perspective.

*   **LLM-as-a-Judge:**
    *   **Use with caution:** While seemingly intuitive, using a separate LLM to judge another LLM's output has significant pitfalls, including potential biases (length, position), inconsistency, and dependence on the judge LLM's own capabilities and prompt sensitivity.
    *   **Best for:** Exploratory analysis or when human evaluation is impractical, but **only** with careful controls like pairwise comparison, randomization, rubric-based prompts, and multi-run aggregation as demonstrated. It's valuable for understanding subjective quality aspects but should be validated against human judgment.

**Combining Approaches:**

Often, the most effective evaluation strategy combines multiple methods. Use traditional metrics for quantitative similarity checks, and use LLM-as-judge cautiously for insights into subjective quality or when references are difficult to obtain, always being mindful of its limitations and implementing controls.

Ultimately, **human evaluation** remains the gold standard for assessing true quality, helpfulness, and factual accuracy, especially for critical applications. Automated metrics and LLM judges can serve as valuable proxies and filters, but should ideally be correlated with human judgments.

---

$$END$$

---

### Please answer the questions below to complete the experiment:




Consider the following statements to answer the question given below:

1. BLEU mainly focuses on semantic meaning rather than word overlap.

2. ROUGE mainly checks how much of the reference content is covered by the generated output.

3. METEOR considers synonyms and word stems in addition to exact word matches.

4. LLM-as-a-Judge does not require a clear rubric or evaluation criteria.

In [ ]:
#@title Select the option that correctly identifies the True or False status of each of the statements given above { run: "auto", form-width: "500px", display-mode: "form" }
Answer = "" #@param ["", "1-True 2-False 3-True 4-False", "1-False 2-True 3-True 4-False", "1-True 2-True 3-False 4-True", "1-False 2-False 3-False 4-True"]

In [ ]:
#@title How was the experiment? { run: "auto", form-width: "500px", display-mode: "form" }
Complexity = "" #@param ["","Too Simple, I am wasting time", "Good, But Not Challenging for me", "Good and Challenging for me", "Was Tough, but I did it", "Too Difficult for me"]


In [ ]:
#@title If it was too easy, what more would you have liked to be added? If it was very difficult, what would you have liked to have been removed? { run: "auto", display-mode: "form" }
Additional = "" #@param {type:"string"}


In [ ]:
#@title Can you identify the concepts from the lecture which this experiment covered? { run: "auto", vertical-output: true, display-mode: "form" }
Concepts = "" #@param ["","Yes", "No"]


In [ ]:
#@title  Text and image description/explanation and code comments within the experiment: { run: "auto", vertical-output: true, display-mode: "form" }
Comments = "" #@param ["","Very Useful", "Somewhat Useful", "Not Useful", "Didn't use"]


In [ ]:
#@title Mentor Support: { run: "auto", vertical-output: true, display-mode: "form" }
Mentor_support = "" #@param ["","Very Useful", "Somewhat Useful", "Not Useful", "Didn't use"]


In [ ]:
#@title Run this cell to submit your notebook for grading { vertical-output: true }
try:
  if submission_id:
      return_id = submit_notebook()
      if return_id : submission_id = return_id
  else:
      print("Please complete the setup first.")
except NameError:
  print ("Please complete the setup first.")